## tl;dr

대학알리미 대학별 학과정보 API의 2025년 전체 60,919행을 수집했다. 일반·전문·특수대학원은 22,185행, 대학원명은 1,581개다. EDSS 취업통계 2023–2024년 미연결 대학원 학교–연도 2,435개 중 2,286개(93.8809%)가 대학원명·종류·지역으로 유일하게 연결됐고, 2,281개는 학과도 하나 이상 겹쳤다. 다만 API에는 대학원별 식별자나 EDSS 개방ID가 없고 기준연도도 다르므로 개방ID는 대입하지 않는다.

## Context & Methods

목적은 새 API가 남은 대학원 식별자 공백을 얼마나 좁힐 수 있는지 검증하는 것이다. API 학교명을 유니코드·공백·기호 기준으로 정규화하고, 학교종류와 시도를 함께 비교했다. 끝의 캠퍼스·지역 괄호는 보조 규칙에서만 제거했다. 동일 맥락에서 API 학교명이 하나일 때만 이름 후보로 기록하고 학과 집합의 교집합과 Jaccard를 계산했다.

### Key Assumptions

- 2025년 학교명은 2023–2024년 학교의 후속 검증 자료로만 사용한다.
- 학과명 겹침은 보조 근거이며 학과 개폐·명칭 변경을 감안한다.
- 이름 후보는 공식 식별자 교차표가 아니므로 후보 및 정식 개방ID 열은 비워 둔다.

## Data

In [1]:
import csv
import json
from collections import Counter
from pathlib import Path

repository_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
collection_path = repository_root / 'data/metadata/academyinfo_school_major_collection.json'
match_path = repository_root / 'data/metadata/edss_academyinfo_graduate_name_match.json'
candidate_path = repository_root / 'data/metadata/edss_academyinfo_graduate_name_candidates.csv'

collection = json.loads(collection_path.read_text(encoding='utf-8'))
match = json.loads(match_path.read_text(encoding='utf-8'))
with candidate_path.open(encoding='utf-8-sig', newline='') as handle:
    candidates = list(csv.DictReader(handle))

assert len(candidates) == match['graduate_identity_count']
assert all(row['candidate_open_id'] == '' for row in candidates)
assert all(row['canonical_open_id_imputed'] == 'false' for row in candidates)
{'collection_status': collection['status'], 'match_status': match['status'], 'candidate_rows': len(candidates)}

{'collection_status': 'review_required',
 'match_status': 'review_required',
 'candidate_rows': 2435}

## Results

In [2]:
year_summary = collection['year_summaries'][0]
summary = {
    'api_rows': year_summary['row_count'],
    'api_exact_duplicate_rows': year_summary['exact_duplicate_row_count'],
    'graduate_rows': year_summary['graduate_row_count'],
    'graduate_schools': year_summary['graduate_school_count'],
    'edss_graduate_school_year_identities': match['graduate_identity_count'],
    'edss_source_rows': match['source_row_count'],
    'unique_name_context_matches': match['unique_name_context_match_count'],
    'unique_name_context_match_rate': match['unique_name_context_match_rate'],
    'positive_department_overlap': match['positive_department_overlap_count'],
    'open_ids_imputed': match['canonical_open_id_imputed_row_count'],
}
summary

{'api_rows': 60919,
 'api_exact_duplicate_rows': 284,
 'graduate_rows': 22185,
 'graduate_schools': 1581,
 'edss_graduate_school_year_identities': 2435,
 'edss_source_rows': 19477,
 'unique_name_context_matches': 2286,
 'unique_name_context_match_rate': 0.938809,
 'positive_department_overlap': 2281,
 'open_ids_imputed': 0}

In [3]:
status_by_kind = Counter((row['edss_school_kind'], row['match_status']) for row in candidates)
sorted((kind, status, count) for (kind, status), count in status_by_kind.items())

[('일반대학원', 'candidate_unique_name_context', 341),
 ('일반대학원', 'unmatched_name_context', 34),
 ('전문대학원', 'candidate_unique_name_context', 386),
 ('전문대학원', 'unmatched_name_context', 15),
 ('특수대학원', 'candidate_unique_name_context', 1559),
 ('특수대학원', 'unmatched_name_context', 100)]

In [4]:
unmatched = [
    {key: row[key] for key in ('_panel_year', 'edss_school_name', 'edss_school_kind', 'edss_province')}
    for row in candidates
    if row['match_status'] == 'unmatched_name_context'
]
{'unmatched_count': len(unmatched), 'sample': unmatched[:10]}

{'unmatched_count': 149,
 'sample': [{'_panel_year': '2023',
   'edss_school_name': '강릉원주대학교대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '강원'},
  {'_panel_year': '2023',
   'edss_school_name': '강릉원주대학교대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '강원'},
  {'_panel_year': '2023',
   'edss_school_name': '건국대학교 일반대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '충북'},
  {'_panel_year': '2023',
   'edss_school_name': '경주대학교일반대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '경북'},
  {'_panel_year': '2023',
   'edss_school_name': '고려대학교 일반대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '세종'},
  {'_panel_year': '2023',
   'edss_school_name': '공주대학교대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '충남'},
  {'_panel_year': '2023',
   'edss_school_name': '군산대학교대학원',
   'edss_school_kind': '일반대학원',
   'edss_province': '전북'},
  {'_panel_year': '2023',
   'edss_school_name': '금오공과대학교대학원',
   'edss_school_kind': '일반대학원',
   'edss_province'

## Takeaways

- API 신청과 수집은 정상 완료됐고 대학원 단위 학교명이 실제로 제공된다.
- 2,435개 미연결 대학원 학교–연도 중 2,286개를 이름·종류·지역의 유일 후보로 좁힐 수 있다.
- 149개는 2025년 API 이름과 연결되지 않아 폐지·통합·명칭 변경 이력 검토가 필요하다.
- 원천의 완전 중복 284행을 보존·기록했으며 downstream 집계에서는 중복 통제가 필요하다.
- 대학원별 식별자가 없으므로 이 결과만으로 개방ID를 확정하거나 대입하지 않는다.